# 04 — Scenario compare (Phase 2)

What-if rebalancing on top of the Phase 1 diagnostic. Express a proposed
book via the edit helpers (`drop_portfolio`, `rebalance_into`, `merge_into`,
`set_weights`) and diff it against the current book — every Phase 1 metric
is recomputed on both sides and surfaced as deltas.

## Setup — load real book + market data

In [1]:
from datetime import date
from pathlib import Path

import pandas as pd

from hailmary.allocation.book_config import MGMT_FEES_ANNUAL, ROLES
from hailmary.allocation.portfolios import Role, from_parsed
from hailmary.allocation.statements import parse_statement
from hailmary.allocation.returns import last_business_day_on_or_before
from hailmary.allocation.scenarios import (
    Scenario, drop_portfolio, merge_into, rebalance_into, scenario_compare, set_weights,
)
from hailmary.data.providers import YahooFinanceProvider

STATEMENT_PATH = Path('../../data/statements/2026-04 StashAway Monthly Statement.pdf')
START = date(2022, 1, 1)
END = last_business_day_on_or_before(date.today())

parsed = parse_statement(STATEMENT_PATH)
portfolios = [
    from_parsed(
        p,
        roles=ROLES[p.name],
        metadata={'management_fee_annual': MGMT_FEES_ANNUAL.get(p.name, 0.0)},
    )
    for p in parsed if p.name in ROLES
]
holding = [p for p in portfolios if Role.HOLDING in p.roles]
tickers = sorted({
    h.metadata.ticker for p in holding for h in p.holdings
    if not h.metadata.ticker.startswith('CASH_')
})
provider = YahooFinanceProvider()
returns = provider.get_returns(tickers, START, END)
fx_bars = provider.get_bars(['USDSGD=X'], START, END)
fx_series_usd_sgd = fx_bars.xs('USDSGD=X', level=0)['close']
print(f'{len(portfolios)} portfolios loaded, {len(holding)} HOLDING-tagged, '
      f'window {START}..{END}')

2026-05-30 12:09:49.788 | DEBUG    | hailmary.allocation.statements:get:169 - Statement cache hit for 2026-04 StashAway Monthly Statement.pdf


2026-05-30 12:09:49.790 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=2d7b97eadd63


2026-05-30 12:09:49.820 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=0d6bb24230e4


15 portfolios loaded, 15 HOLDING-tagged, window 2022-01-01..2026-05-29


In [2]:
def headline(diff):
    d = diff.deltas
    print(f'Combined book changes ({diff.current.label}  ->  {diff.proposed.label}):')
    print(f'  Sharpe Δ:    {d.book_sharpe_delta:+.3f}')
    print(f'  Ann ret Δ:   {d.book_ann_return_delta:+.2%}')
    print(f'  Vol Δ:       {d.book_ann_vol_delta:+.2%}')
    print(f'  Max DD Δ:    {d.book_max_dd_delta:+.2%}')
    print(f'  AUM Δ:       {d.book_aum_delta:+,.0f} SGD')
    if d.redundancy_appeared:
        print(f'  Redundancy appeared:    {d.redundancy_appeared}')
    if d.redundancy_disappeared:
        print(f'  Redundancy disappeared: {d.redundancy_disappeared}')

cur = Scenario('current book', tuple(portfolios))
kw = dict(returns=returns, fx_series_usd_sgd=fx_series_usd_sgd, align_window=True)

## Scenario A — What if I dropped Crypto?

Removes the Crypto sleeve entirely. The freed capital is just *gone* from the
book (use `rebalance_into` if you want to redeploy it). Useful for seeing
how much risk Crypto is contributing.

In [3]:
proposed = drop_portfolio(portfolios, 'Crypto')
diff_a = scenario_compare(cur, Scenario('drop Crypto', proposed), **kw)
headline(diff_a)

Combined book changes (current book  ->  drop Crypto):
  Sharpe Δ:    +0.178
  Ann ret Δ:   +0.40%
  Vol Δ:       -0.70%
  Max DD Δ:    +0.83%
  AUM Δ:       -69,775 SGD


In [4]:
print('Asset-class exposure shift:')
diff_a.deltas.exposure_delta['asset_class'].head(10)

Asset-class exposure shift:


,bucket,value_cur,weight_cur,value_prop,weight_prop,weight_delta,value_delta
3,Crypto,107445.47,0.036388,53190.66,0.018354,-0.018033,-54254.81
1,Cash,2307268.54,0.781382,2306720.41,0.795969,0.014587,-548.13
4,Equity,489817.89,0.165882,489817.89,0.169019,0.003137,0.00
2,Commodity,26155.95,0.008858,26155.95,0.009026,0.000168,0.00
0,Bond,22118.26,0.007491,22118.26,0.007632,0.000142,0.00


## Scenario B — What if I moved Crypto into BlackRock?

`rebalance_into` moves Crypto's whole `total_value` into BlackRock
at BlackRock's current composition. Crypto disappears; BlackRock gets bigger.
Both are USD sleeves — same-currency rotations only (cross-currency rotations
raise `ScenarioEditError`; convert FX first if needed).

In [5]:
proposed = rebalance_into(portfolios, 'Crypto', 'BlackRock')
diff_b = scenario_compare(cur, Scenario('Crypto -> BlackRock', proposed), **kw)
headline(diff_b)

Combined book changes (current book  ->  Crypto -> BlackRock):
  Sharpe Δ:    +0.189
  Ann ret Δ:   +0.51%
  Vol Δ:       -0.60%
  Max DD Δ:    +0.67%
  AUM Δ:       +0 SGD


## Scenario C — What if I merged Energy + Utilities + HDY into one sleeve?

Value-weighted union of the three customs into a single 'Custom Equity Sleeve'.
Same total exposure — just consolidated administratively.

In [6]:
proposed = merge_into(
    portfolios,
    names=['Energy', 'Utilities', 'High Dividend Yield'],
    into='Custom Equity Sleeve',
)
diff_c = scenario_compare(cur, Scenario('merge customs', proposed), **kw)
headline(diff_c)

Combined book changes (current book  ->  merge customs):
  Sharpe Δ:    -0.000
  Ann ret Δ:   -0.00%
  Vol Δ:       -0.00%
  Max DD Δ:    -0.00%
  AUM Δ:       +0 SGD


## Scenario D — What if I shifted Crypto weights 50/50 BTC/ETH?

`set_weights` replaces one sleeve's weights. Requires explicit weight for
every existing holding (pass 0.0 to zero one out).

In [7]:
crypto = next(p for p in portfolios if p.name == 'Crypto')
print('Current Crypto holdings:')
for h in crypto.holdings:
    print(f'  {h.stashaway_id:<10} weight={h.weight:.3f}')

Current Crypto holdings:
  FBTC       weight=0.506
  FETH       weight=0.484
  CASH_USD   weight=0.000
  CASH_SGD   weight=0.010


In [8]:
# Build a 50/50 BTC/ETH proposal (zero out any other holdings)
current_weights = {h.stashaway_id: h.weight for h in crypto.holdings}
new_weights = {sid: 0.0 for sid in current_weights}
if 'FBTC' in new_weights: new_weights['FBTC'] = 0.5
if 'FETH' in new_weights: new_weights['FETH'] = 0.5
# Sanity: weights sum to 1
assert abs(sum(new_weights.values()) - 1.0) < 1e-9, new_weights

proposed = set_weights(portfolios, 'Crypto', new_weights)
diff_d = scenario_compare(cur, Scenario('Crypto 50/50 BTC/ETH', proposed), **kw)
headline(diff_d)

Combined book changes (current book  ->  Crypto 50/50 BTC/ETH):
  Sharpe Δ:    -0.002
  Ann ret Δ:   -0.00%
  Vol Δ:       +0.01%
  Max DD Δ:    -0.02%
  AUM Δ:       +0 SGD


## Compare all four scenarios side-by-side

In [9]:
import pandas as pd
rows = []
for label, diff in [
    ('A: drop Crypto', diff_a),
    ('B: Crypto -> SI', diff_b),
    ('C: merge customs', diff_c),
    ('D: 50/50 BTC/ETH', diff_d),
]:
    d = diff.deltas
    rows.append({
        'scenario': label,
        'Sharpe Δ': d.book_sharpe_delta,
        'AnnRet Δ': d.book_ann_return_delta,
        'Vol Δ':    d.book_ann_vol_delta,
        'MaxDD Δ':  d.book_max_dd_delta,
        'AUM Δ':    d.book_aum_delta,
        'red. appeared':    len(d.redundancy_appeared),
        'red. disappeared': len(d.redundancy_disappeared),
    })
summary = pd.DataFrame(rows).set_index('scenario')
summary.style.format({
    'Sharpe Δ': '{:+.3f}', 'AnnRet Δ': '{:+.2%}',
    'Vol Δ': '{:+.2%}', 'MaxDD Δ': '{:+.2%}', 'AUM Δ': '{:+,.0f}',
}, na_rep='-')

,Sharpe Δ,AnnRet Δ,Vol Δ,MaxDD Δ,AUM Δ,red. appeared,red. disappeared
scenario,,,,,,,
A: drop Crypto,+0.178,+0.40%,-0.70%,+0.83%,"-69,775",0,0
B: Crypto -> SI,+0.189,+0.51%,-0.60%,+0.67%,+0,0,0
C: merge customs,-0.000,-0.00%,-0.00%,-0.00%,+0,0,0
D: 50/50 BTC/ETH,-0.002,-0.00%,+0.01%,-0.02%,+0,0,0


## Want HTML?

Chunk 3 (`render_scenario_report`) is still TODO. Once built, every `diff`
above can be exported to a self-contained side-by-side HTML with the same
styling as `reports/allocation_diagnostic.html`.